`InsightAgent`. ** A natural-language *analytics agent* that answers business questions over a **real** online-retail database. We picked analytics on purpose: SQL gives the agent **real, verifiable feedback** (a query either errors, or returns checkable numbers). As you'll see, that "grounding" is the single most important ingredient for reflection that actually works.

# The Journey


1. Reasoning Strategies: CoT, ReAct, Plan-and-Solve, ReWOO, self-consistency.
2. We will implement ReAct Agent
3. Implement Reflection (self-correction, LLM as Judge, Tool baed, CRITIQUE, Reflexion Loop)
4. Memory - Short-term, episodic, semantic, poisoning.

| | **CoT** | **ReAct** | **Plan-and-Solve** | **ReWOO** | **Self-Consistency** |
|---|---|---|---|---|---|
| Uses external tools? | ❌ | ✅ (core) | ✅ usually | ✅ (planned) | ❌ |
| Decides steps… | one pass | **reactively, one at a time** | **all up front** | **all up front (as vars)** | n/a (samples chains) |
| Grounded in real data? | ❌ | ✅ | ✅ | ✅ | ❌ |
| Adapts mid-task? | ❌ | ✅ strong | ⚠️ re-plan only | ❌ | ❌ |
| Relative cost | 💲 | 💲💲💲 (re-sends transcript) | 💲💲 | 💲 (fewest calls) | 💲💲💲 (N samples) |
| Best for | self-contained math/logic | open-ended lookups | many ordered steps | tool-heavy, stable plans | one-true-answer problems |
| Main failure mode | confident hallucination | thrashing / looping | flawed plan, executed faithfully | can't adapt to surprises | majority can still be wrong |

In [ ]:
!pip install truststore

In [ ]:
# --- Setup: load the API key + a tidy printer for class output ---
import os, json, re, time
from dotenv import load_dotenv
import textwrap

import truststore                 # trust the OS cert store -> works behind a VPN / corporate proxy
truststore.inject_into_ssl()

def pretty_print(*args, width=80):
    '''Readable console output for class: reflow long PROSE to `width` columns, but leave
    multi-line / column-aligned text (tables, schemas, SQL results) untouched so nothing
    gets garbled.'''
    text = " ".join(str(a) for a in args)
    core = text.strip("\n")
    if "\n" in core or re.search(r"\S  +\S", core):        # pre-formatted -> print as-is
        print(text)
    else:                                                   # prose -> wrap, keep blank-line padding
        lead = "\n" * (len(text) - len(text.lstrip("\n")))
        tail = "\n" * (len(text) - len(text.rstrip("\n")))
        print(lead + textwrap.fill(core, width=width) + tail)

# OpenAI is the provider used throughout. Point this at your own env file.
load_dotenv("openai_key.env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found — check your openai_key.env path.")
pretty_print("API key loaded successfully.")

API key loaded successfully.


In [ ]:
FAST_MODEL   = "gpt-4.1-nano" # $0.4
STRONG_MODEL = "gpt-4.1-mini" # $1.4
EMBED_MODEL  = "text-embedding-3-small"

In [ ]:
# Tiny, shared LLM helpers used everywhere below. Thin wrappers over the OpenAI SDK
# so the *agent logic* stays readable.
from openai import OpenAI
client = OpenAI()

def chat(messages, tools=None, model=FAST_MODEL, temperature=0.0):
    '''One chat-completions call. Returns the assistant *message* object
    (which may contain .content and/or .tool_calls).'''
    kwargs = dict(model=model, messages=messages, temperature=temperature)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    return client.chat.completions.create(**kwargs).choices[0].message

def ask(prompt, system=None, model=FAST_MODEL, temperature=0.0):
    '''Convenience: single-turn text in, text out.'''
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    return chat(msgs, model=model, temperature=temperature).content

def embed(texts):
    '''Embed a string or list of strings -> list[list[float]].'''
    if isinstance(texts, str): texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

pretty_print("helpers ready.")

helpers ready.


In [ ]:
import os, io, ssl, zipfile, sqlite3, urllib.request
import pandas as pd

DB_PATH = "online_retail.db"
ZIP_PATH = "online_retail.zip"          # cache the raw download so rebuilds need no network
URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

In [ ]:
if os.path.exists(ZIP_PATH):
    pretty_print("Using cached zip", ZIP_PATH)
    raw = open(ZIP_PATH, "rb").read()
else:
    pretty_print("Downloading real UCI Online Retail data (~24MB)…")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=120, context=ssl.create_default_context()).read()
    with open(ZIP_PATH, "wb") as f:    # save for future runs
        f.write(raw)
    pretty_print("Saved", ZIP_PATH)
xlsx = zipfile.ZipFile(io.BytesIO(raw)).read("Online Retail.xlsx")
df = pd.read_excel(io.BytesIO(xlsx), engine="openpyxl")
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

Saved online_retail.zip


In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


1. invoice
2. products
3. transactions

In [ ]:
# Normalise the flat file into a small star schema (so the agent must JOIN).
invoices = (df.groupby("InvoiceNo")
              .agg(customer_id=("CustomerID", "first"),
                    invoice_ts=("InvoiceDate", "first"),
                    country=("Country", "first")).reset_index())
invoices["is_cancelled"] = invoices["InvoiceNo"].str.startswith("C").astype(int)
invoices = invoices.rename(columns={"InvoiceNo": "invoice_no"})
invoices["invoice_ts"] = invoices["invoice_ts"].astype(str)
invoices.head()

,invoice_no,customer_id,invoice_ts,country,is_cancelled
0,536365,17850.0,2010-12-01 08:26:00,United Kingdom,0
1,536366,17850.0,2010-12-01 08:28:00,United Kingdom,0
2,536367,13047.0,2010-12-01 08:34:00,United Kingdom,0
3,536368,13047.0,2010-12-01 08:34:00,United Kingdom,0
4,536369,13047.0,2010-12-01 08:35:00,United Kingdom,0


In [ ]:
products = (df.dropna(subset=["Description"])
              .groupby("StockCode")["Description"]
              .agg(lambda s: s.value_counts().index[0]).reset_index())
products.columns = ["stock_code", "description"]
products.head()

,stock_code,description
0,10002,INFLATABLE POLITICAL GLOBE
1,10080,GROOVY CACTUS INFLATABLE
2,10120,DOGGY RUBBER
3,10125,MINI FUNKY DESIGN TAPES
4,10133,COLOURING PENCILS BROWN TUBE


In [ ]:
lines = df[["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]].copy()
lines.columns = ["invoice_no", "stock_code", "quantity", "unit_price"]
lines.head()

,invoice_no,stock_code,quantity,unit_price
0,536365,85123A,6,2.55
1,536365,71053,6,3.39
2,536365,84406B,8,2.75
3,536365,84029G,6,3.39
4,536365,84029E,6,3.39


In [ ]:
# Build (once) a small multi-table SQLite DB from the real dataset, then cache it.
# First run downloads ~24MB and takes ~60-90s; after that it's instant.
import os, io, ssl, zipfile, sqlite3, urllib.request
import pandas as pd

DB_PATH = "online_retail.db"
ZIP_PATH = "online_retail.zip"          # cache the raw download so rebuilds need no network
URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

def build_db(path=DB_PATH):
    if os.path.exists(ZIP_PATH):
        pretty_print("Using cached zip", ZIP_PATH)
        raw = open(ZIP_PATH, "rb").read()
    else:
        pretty_print("Downloading real UCI Online Retail data (~24MB)…")
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        raw = urllib.request.urlopen(req, timeout=120, context=ssl.create_default_context()).read()
        with open(ZIP_PATH, "wb") as f:    # save for future runs
            f.write(raw)
        pretty_print("Saved", ZIP_PATH)
    xlsx = zipfile.ZipFile(io.BytesIO(raw)).read("Online Retail.xlsx")
    df = pd.read_excel(io.BytesIO(xlsx), engine="openpyxl")
    df["InvoiceNo"] = df["InvoiceNo"].astype(str)

    # Normalise the flat file into a small star schema (so the agent must JOIN).
    invoices = (df.groupby("InvoiceNo")
                  .agg(customer_id=("CustomerID", "first"),
                       invoice_ts=("InvoiceDate", "first"),
                       country=("Country", "first")).reset_index())
    invoices["is_cancelled"] = invoices["InvoiceNo"].str.startswith("C").astype(int)
    invoices = invoices.rename(columns={"InvoiceNo": "invoice_no"})
    invoices["invoice_ts"] = invoices["invoice_ts"].astype(str)

    products = (df.dropna(subset=["Description"])
                  .groupby("StockCode")["Description"]
                  .agg(lambda s: s.value_counts().index[0]).reset_index())
    products.columns = ["stock_code", "description"]

    lines = df[["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]].copy()
    lines.columns = ["invoice_no", "stock_code", "quantity", "unit_price"]

    con = sqlite3.connect(path)
    invoices.to_sql("invoices", con, index=False, if_exists="replace")
    products.to_sql("products", con, index=False, if_exists="replace")
    lines.to_sql("line_items", con, index=False, if_exists="replace")
    con.executescript("CREATE INDEX IF NOT EXISTS i_li_inv ON line_items(invoice_no);"
                      "CREATE INDEX IF NOT EXISTS i_li_sc  ON line_items(stock_code);")
    con.commit(); con.close()
    pretty_print("Built", path)

if not os.path.exists(DB_PATH):
    build_db()
else:
    pretty_print("Using cached", DB_PATH)

# Peek at the schema.
con = sqlite3.connect(DB_PATH)
for t in ["invoices", "products", "line_items"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    pretty_print(f"  {t:11s} {n:>8,} rows")
con.close()

Using cached zip online_retail.zip
Built online_retail.db
  invoices      25,900 rows
  products       3,958 rows
  line_items   541,909 rows


In [ ]:
# A question whose answer lives ONLY in our database — the model cannot know it.
QUESTION = "What was the total revenue from customers in the United Kingdom, " \
           "excluding cancelled orders? Give a single number."

samples = [ask(QUESTION + "\n\nReason briefly, end with a single number.",
               temperature=1.0) for _ in range(5)]
for i, s in enumerate(samples):
    pretty_print(f"  sample {i+1}: {s}")
    print('*' * 40, '\n' * 3)
pretty_print("\n→ Look at the spread. Self-consistency picks the *majority*, but here every "
      "sample is ungrounded — the 'majority hallucination' is still a hallucination.")
pretty_print("→ The fix isn't more samples. It's giving the model a TOOL to look it up (§2).")

  sample 1: To determine the total revenue from customers in the United Kingdom excluding canceled orders, I would need specific data such as order details, customer locations, order statuses, and revenue amounts. Since no such data is provided, I cannot perform the calculation.

0
**************************************** 



  sample 2: To determine the total revenue from customers in the United Kingdom, excluding canceled orders, I would need access to the relevant data such as order records including customer location, order status, and revenue amounts.

Since I do not have the specific data provided, I cannot calculate the total revenue directly. Please provide the dataset or the necessary details so I can assist further.

If you have the data, please share it or the summarized figures, and I'll help compute the total revenue.
**************************************** 



  sample 3: I don't have access to the specific dataset or information regarding the orders from the United King

In [ ]:
# --- Plan-and-Solve: ask the model to PLAN first (ReAct itself is built in §2) ---
plan = ask(
    f"Task: {QUESTION}\n\nYou have tools to list tables, inspect a table's schema, "
    "and run SQL on a SQLite e-commerce DB. Do NOT answer yet — just write a short, "
    "numbered PLAN of the steps you'd take.",
    system="You are a data analyst that plans before acting.",
)
pretty_print("PLAN-AND-SOLVE — the plan it would execute:\n", plan)

PLAN-AND-SOLVE — the plan it would execute:
 1. List all available tables in the database to identify relevant tables (e.g., orders, customers, order_items, etc.).
2. Inspect the schema of the tables related to orders and customers to understand their structure and relationships.
3. Identify the table(s) that contain order status or cancellation information to filter out cancelled orders.
4. Write an SQL query to join the relevant tables, filter for customers in the United Kingdom, exclude cancelled orders, and sum the total revenue.
5. Execute the query to obtain the total revenue from UK customers excluding cancelled orders.


# Build ReAct Agent

Defining right tools is winning half the battle

In [ ]:
import sqlite3

def _connect():
    con = sqlite3.connect(DB_PATH)
    return con


def list_tables():
    '''List the tables available in the database.'''
    con = _connect()
    rows = con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    con.close()
    return ", ".join(r[0] for r in rows)


def get_schema(table):
    '''Show columns (name + type) and a couple of sample rows for one table.'''
    con = _connect()
    try:
        cols = con.execute(f"PRAGMA table_info({table})").fetchall()
        if not cols:
            return f"No such table: {table}"
        sample = con.execute(f"SELECT * FROM {table} LIMIT 2").fetchall()
        lines = [f"Table '{table}':"] + [f"  - {c[1]} ({c[2]})" for c in cols]
        lines.append(f"  sample rows: {sample}")
        return "\n".join(lines)
    finally:
        con.close()


def run_sql(query, max_rows=20):
    '''Run a read-only SQL query; return rows as text, or a SQL ERROR string.'''
    con = _connect()
    try:
        cur = con.execute(query)
        if cur.description is None:
            return "OK (statement executed, no rows)."
        colnames = [d[0] for d in cur.description]
        rows = cur.fetchmany(max_rows)
        more = cur.fetchone() is not None
        header = " | ".join(colnames)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows) or "(0 rows)"
        note = f"\n… (truncated at {max_rows} rows)" if more else ""
        return f"{header}\n{body}{note}"
    except Exception as e:
        # Returning the error as a STRING (not raising) is deliberate — it becomes an
        # Observation the agent can REFLECT on. More on that in §3.
        return f"SQL ERROR: {type(e).__name__}: {e}"
    finally:
        con.close()

In [ ]:
pretty_print(list_tables())
pretty_print()
pretty_print(get_schema("invoices"))
pretty_print()
pretty_print(run_sql("SELECT country, COUNT(*) n FROM invoices GROUP BY country ORDER BY n DESC LIMIT 3"))
pretty_print()
pretty_print(run_sql("SELECT * FROM no_such_table"))   # see the friendly error-as-observation

invoices, line_items, products

Table 'invoices':
  - invoice_no (TEXT)
  - customer_id (REAL)
  - invoice_ts (TEXT)
  - country (TEXT)
  - is_cancelled (INTEGER)
  sample rows: [('536365', 17850.0, '2010-12-01 08:26:00', 'United Kingdom', 0), ('536366', 17850.0, '2010-12-01 08:28:00', 'United Kingdom', 0)]

country | n
United Kingdom | 23494
Germany | 603
France | 461

SQL ERROR: OperationalError: no such table: no_such_table


In [ ]:
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "list_tables", "description": "List all tables in the database.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "get_schema", "description": "Show columns and sample rows for one table.",
        "parameters": {"type": "object",
                       "properties": {"table": {"type": "string"}},
                       "required": ["table"]}}},
    {"type": "function", "function": {
        "name": "run_sql", "description": "Run a read-only SQLite query and return rows.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
]
TOOLS = {"list_tables": lambda: list_tables(),
         "get_schema":  get_schema,
         "run_sql":     run_sql}
pretty_print("registered tools:", list(TOOLS))

registered tools: ['list_tables', 'get_schema', 'run_sql']


In [ ]:
import json

SYSTEM_REACT = (
    "You are InsightAgent, a data analyst for an online-retail store. "
    "Answer the user's question by exploring the SQLite database with your tools. "
    "ALWAYS inspect the schema before writing SQL. Reason step by step. "
    "When you are confident, give the final answer clearly, including the number."
)


def run_react(question, system=SYSTEM_REACT, model=FAST_MODEL, max_steps=8, verbose=True):

      messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]


      for steps in range(max_steps):
        msg = chat(messages, tools=TOOL_SCHEMAS, model=model)
        messages.append(msg.model_dump(exclude_none=True))   # remember what the agent said


        if msg.content and verbose:                          # the THOUGHT
          pretty_print(f"🤔 {msg.content.strip()}")

        if not msg.tool_calls:                               # no action -> it's done
            if verbose: pretty_print(f"\n✅ FINAL ANSWER:\n{msg.content}")
            return msg.content


        for tc in msg.tool_calls:                            # the ACTION(s)
          name = tc.function.name
          args = json.loads(tc.function.arguments or "{}")
          obs = TOOLS[name](**args)                         # the OBSERVATION
          if verbose:
              pretty_print(f"  🛠️  {name}({args})")
              pretty_print("  👀 " + str(obs)[:350].replace("\n", "\n     "))
          messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(obs)})
      return "⚠️ Stopped: hit max_steps (possible loop)."

In [ ]:
QUESTION

'What was the total revenue from customers in the United Kingdom, excluding cancelled orders? Give a single number.'

In [ ]:
answer = run_react(QUESTION)

  🛠️  list_tables({})
  👀 invoices, line_items, products
  🛠️  get_schema({'table': 'orders'})
  👀 No such table: orders
  🛠️  get_schema({'table': 'invoices'})
  👀 Table 'invoices':
       - invoice_no (TEXT)
       - customer_id (REAL)
       - invoice_ts (TEXT)
       - country (TEXT)
       - is_cancelled (INTEGER)
       sample rows: [('536365', 17850.0, '2010-12-01 08:26:00', 'United Kingdom', 0), ('536366', 17850.0, '2010-12-01 08:28:00', 'United Kingdom', 0)]
  🛠️  get_schema({'table': 'line_items'})
  👀 Table 'line_items':
       - invoice_no (TEXT)
       - stock_code (TEXT)
       - quantity (INTEGER)
       - unit_price (REAL)
       sample rows: [('536365', '85123A', 6, 2.55), ('536365', '71053', 6, 3.39)]
  🛠️  run_sql({'query': "SELECT invoice_no, country, is_cancelled FROM invoices WHERE country='United Kingdom' AND is_cancelled=0;"})
  👀 invoice_no | country | is_cancelled
     536365 | United Kingdom | 0
     536366 | United Kingdom | 0
     536367 | United Kingdom | 

  🛠️  run_sql({'query': "SELECT i.invoice_no, SUM(li.quantity * li.unit_price) AS total_revenue FROM invoices i JOIN line_items li ON i.invoice_no = li.invoice_no WHERE i.country='United Kingdom' AND i.is_cancelled=0 GROUP BY i.invoice_no;"})


structured_outputs and parallized tool-call (some production oriented steps)

1. Thrashing: max_steps + hashing(function + args)

In [ ]:
# A minimal loop/thrash detector you can drop into any ReAct loop. No API needed.
import hashlib, json

def call_signature(name, args):
    '''Stable short hash of a tool call so we can spot exact repeats.'''
    return hashlib.sha1((name + "::" + json.dumps(args, sort_keys=True)).encode()).hexdigest()[:10]

def detect_thrash(calls, repeat_limit=2):
    '''calls = list of (name, args). Returns the call that repeats too often, else None.'''
    seen = {}
    for name, args in calls:
        sig = call_signature(name, args)
        seen[sig] = seen.get(sig, 0) + 1
        if seen[sig] >= repeat_limit:
            return name, args, seen[sig]
    return None


sim = [("run_sql", {"query": "SELECT * FROM order"}),   # 'order' is reserved -> errors
       ("run_sql", {"query": "SELECT * FROM order"}),
       ("get_schema", {"table": "invoices"})]
pretty_print("Thrash detected on:", detect_thrash(sim, repeat_limit=2))
pretty_print("→ When this fires, inject 'You already tried that and it failed — try a DIFFERENT "
      "approach' instead of letting the agent burn steps.")

Thrash detected on: ('run_sql', {'query': 'SELECT * FROM order'}, 2)
→ When this fires, inject 'You already tried that and it failed — try a
DIFFERENT approach' instead of letting the agent burn steps.


In [ ]:
# reflection step

import re

def clean_sql(text):
    '''Strip markdown fences / prose the model sometimes adds around SQL.'''
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S)
    sql = m.group(1) if m else text
    return sql.strip().rstrip(";").strip()

SCHEMA_TEXT = None  # filled lazily so non-LLM cells don't need it

def generate_sql(question, hints=""):
    global SCHEMA_TEXT
    if SCHEMA_TEXT is None:
        SCHEMA_TEXT = "\n\n".join(get_schema(t) for t in ["invoices", "products", "line_items"])
    prompt = (f"Schema:\n{SCHEMA_TEXT}\n\n{hints}\n\n"
              f"Write ONE SQLite query that answers: {question}\nReturn ONLY SQL.")
    return clean_sql(ask(prompt, system="You write correct SQLite queries."))


def answer_with_grounded_reflection(question, max_tries=3, verbose=True):
    hints = ""
    sql = result = None
    for attempt in range(1, max_tries + 1):
        sql = generate_sql(question, hints)
        result = run_sql(sql)
        if verbose:
            pretty_print(f"[attempt {attempt}] {sql}")
        if not str(result).startswith("SQL ERROR"):
            if verbose: pretty_print(f"  ✅ {str(result).splitlines()[0]} …")
            return sql, result
        if verbose: pretty_print(f"  ❌ {result}\n  ↺ reflecting on the real error and revising…")
        hints = (f"Your previous query:\n{sql}\nfailed with:\n{result}\n"
                 "Diagnose the cause from the schema and fix it.")
    return sql, result

In [ ]:
demo_q = ("Total revenue (quantity * unit_price) for line items, using the column "
          "named 'price' for the unit price.")   # 'price' does not exist -> error -> reflect
sql, res = answer_with_grounded_reflection(demo_q)
pretty_print("\nFinal SQL:\n", sql, "\nResult:", res)

[attempt 1] SELECT SUM(quantity * price) AS total_revenue
FROM line_items
  ❌ SQL ERROR: OperationalError: no such column: price
  ↺ reflecting on the real error and revising…
[attempt 2] SELECT SUM(quantity * unit_price) AS total_revenue
FROM line_items
  ✅ total_revenue …

Final SQL:
 SELECT SUM(quantity * unit_price) AS total_revenue
FROM line_items 
Result: total_revenue
9747747.93400317


- **Position bias** — prefers whichever answer appears *first* (or last). Measured by *positional consistency*; even strong judges drift.
- **Verbosity bias** — longer answers score higher regardless of correctness.
- **Self-preference (self-bias)** — a model rates *its own* outputs more kindly. (Why we used a *different, stronger* judge.)
- **Leniency bias** — judges over-grade; everything "looks fine."

In [ ]:
# gpt-5-nano
# claude-haiku-4.5
# claude-opus-4.8
# gpt-4.1-mini


# reviewer is gpt-5-nano

In [ ]:
# Demonstrate POSITION BIAS in a pairwise judge, then fix it by requiring agreement
# across BOTH orderings. (One of the answers is correct; one ignores cancellations.)
PAIRWISE = '''Which SQL better answers the question? Reply with ONLY "A" or "B".
Question: {q}
Answer A:
{a}
Answer B:
{b}'''

def pairwise_pick(q, a, b, model=STRONG_MODEL):
    out = (ask(PAIRWISE.format(q=q, a=a, b=b), model=model) or "").strip().upper()
    return "A" if out.startswith("A") else "B"

qj   = "Total revenue excluding cancelled orders."
good = ("SELECT SUM(li.quantity*li.unit_price) FROM line_items li "
        "JOIN invoices i ON i.invoice_no=li.invoice_no WHERE i.is_cancelled=0")
bad  = "SELECT SUM(quantity*unit_price) FROM line_items"     # ignores cancellations
first  = pairwise_pick(qj, good, bad)   # good is A -> want "A"
second = pairwise_pick(qj, bad, good)   # good is B -> want "B"
pretty_print(f"good-as-A -> judge said: {first}   (correct = A)")
pretty_print(f"good-as-B -> judge said: {second}   (correct = B)")


good-as-A -> judge said: A   (correct = A)
good-as-B -> judge said: B   (correct = B)


LLM as a judge also tends to prefer the options that come in the last!!

### Reflexion

Golden set : set of Q/A hand curated, toughest questions on which LLM generally tends to struggle

In [ ]:
def reflexion_sql(question, evaluate, max_trials=3, verbose=True):
    lessons = []                                    # <- episodic memory of verbal self-reflections
    sql = result = None


    for trial in range(1, max_trials + 1):
      memo = ("\nLessons from your past attempts (do NOT repeat these mistakes):\n"
              + "\n".join(f"- {l}" for l in lessons)) if lessons else ""
      sql = generate_sql(question, memo)          # ACTOR (reuses the §3.1 generator)
      result = run_sql(sql)                        # run it
      ok, feedback = evaluate(sql, result)         # EVALUATOR (a verifier, not just "did it run?")
      if verbose:
          pretty_print(f"[trial {trial}] ok={ok}  sql={str(sql)[:80]}")
      if ok:
          return sql, result, lessons

      lesson = ask(
              f"Schema:\n{SCHEMA_TEXT}\n\nQuestion: {question}\nYour SQL:\n{sql}\n"
              f"It was WRONG: {feedback}\n"
              "Using the schema above, write ONE concrete, reusable lesson that names the exact "
              "fix. Be specific — no generic advice.")  # SELF-REFLECTION (grounded)
      lessons.append(lesson.strip())
      if verbose: pretty_print(f"   💡 lesson: {lesson.strip()}")

    return sql, result, lessons


GOLD_REVENUE = float(run_sql(
    "SELECT ROUND(SUM(li.quantity*li.unit_price),2) AS revenue FROM line_items li "
    "JOIN invoices i ON i.invoice_no=li.invoice_no WHERE i.is_cancelled=0").split("\n")[-1])

def revenue_evaluator(sql, result):
    if str(result).startswith("SQL ERROR"):
        return False, str(result)                    # a hard error is its own grounded feedback
    nums = re.findall(r"-?\d+\.?\d*", result.split("\n")[-1])
    got = float(nums[-1]) if nums else None
    if got is None:
        return False, "Your query did not return a single numeric total."
    if abs(got - GOLD_REVENUE) < 1:
        return True, ""
    return False, (f"Your total is {got:,.2f}, but the verified correct total is "
                   f"{GOLD_REVENUE:,.2f}. Some rows are being counted that should be "
                   "excluded -- re-examine the schema for what distinguishes them.")

sql, res, lessons = reflexion_sql(
    "What is the total revenue across all line items? (revenue = quantity * unit_price)",
    revenue_evaluator)

pretty_print("\nFinal SQL:", sql, "\nResult:", str(res)[:120])
pretty_print("Accumulated lessons (now reusable episodic memory):", lessons)


[trial 1] ok=False  sql=SELECT SUM(quantity * unit_price) AS total_revenue
FROM line_items
   💡 lesson: **Lesson:**  
Exclude line items associated with cancelled invoices by adding a join to the `invoices` table and filtering out `is_cancelled = 1`.  

**Concrete fix:**  
```sql
SELECT SUM(li.quantity * li.unit_price) AS total_revenue
FROM line_items li
JOIN invoices i ON li.invoice_no = i.invoice_no
WHERE i.is_cancelled = 0;
```
[trial 2] ok=True  sql=SELECT SUM(li.quantity * li.unit_price) AS total_revenue
FROM line_items li
JOIN

Final SQL: SELECT SUM(li.quantity * li.unit_price) AS total_revenue
FROM line_items li
JOIN invoices i ON li.invoice_no = i.invoice_no
WHERE i.is_cancelled = 0 
Result: total_revenue
10644560.42400402
Accumulated lessons (now reusable episodic memory): ['**Lesson:**  \nExclude line items associated with cancelled invoices by adding a join to the `invoices` table and filtering out `is_cancelled = 1`.  \n\n**Concrete fix:**  \n```sql\nSELECT SUM(li.quantity 

# Memory